# 02 — Chunking Strategies: Como Dividir Documentos

A estrategia de chunking afeta diretamente a qualidade do RAG.
Um bom chunking garante que:
1. Cada chunk tenha significado independente
2. Contexto importante nao seja dividido
3. Chunks nao sejam muito grandes (perdem especificidade) nem pequenos (perdem contexto)

## Estrategias que vamos comparar

| Estrategia | Quando Usar | Pros | Contras |
|-----------|------------|------|----------|
| Fixed-size | Prototipagem | Simples | Corta no meio de frases |
| Recursive | Maioria dos casos | Respeita estrutura | Um pouco mais lento |
| Semantic | Documentos sem estrutura | Melhor qualidade | Precisa de embedding |
| Document-aware | Markdown/HTML | Respeita headings | Especifico por formato |

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

from src.utils.chunking import (
    fixed_size_chunk, recursive_chunk, chunk_with_overlap,
    semantic_chunk, compare_strategies, Chunk
)

model = SentenceTransformer('all-MiniLM-L6-v2')

# Texto de exemplo longo
texto_exemplo = Path('../data/sample_docs/rag_fundamentals.md').read_text(encoding='utf-8')
print(f'Texto: {len(texto_exemplo)} chars, {len(texto_exemplo.split())} palavras')

## 3.1 Fixed-Size Chunking

In [ ]:
chunks_fixed = fixed_size_chunk(texto_exemplo, chunk_size=500, overlap=50)

print(f'Fixed-size (500 chars, 50 overlap):')
print(f'  Numero de chunks: {len(chunks_fixed)}')
print(f'  Chars medios: {sum(len(c.text) for c in chunks_fixed)/len(chunks_fixed):.0f}')
print(f'  Tokens medios: {sum(c.token_count for c in chunks_fixed)/len(chunks_fixed):.0f}')
print(f'\nPrimeiro chunk:')
print(repr(chunks_fixed[0].text[:200]))
print(f'\nSegundo chunk (mostrando overlap):')
print(repr(chunks_fixed[1].text[:200]))

## 3.2 Recursive Character Splitting

In [ ]:
chunks_recursive = recursive_chunk(texto_exemplo, chunk_size=500, overlap=50)

print(f'Recursive split (500 chars, 50 overlap):')
print(f'  Numero de chunks: {len(chunks_recursive)}')
print(f'  Chars medios: {sum(len(c.text) for c in chunks_recursive)/len(chunks_recursive):.0f}')
print(f'  Tokens medios: {sum(c.token_count for c in chunks_recursive)/len(chunks_recursive):.0f}')
print(f'\nPrimeiro chunk:')
print(repr(chunks_recursive[0].text[:300]))

## 3.3 Semantic Chunking

In [ ]:
def embed_fn(texts):
    return model.encode(texts, normalize_embeddings=True).tolist()

chunks_semantic = semantic_chunk(
    texto_exemplo,
    embed_fn=embed_fn,
    similarity_threshold=0.7,
    min_chunk_size=200,
)

print(f'Semantic chunking (threshold=0.7):')
print(f'  Numero de chunks: {len(chunks_semantic)}')
print(f'  Chars medios: {sum(len(c.text) for c in chunks_semantic)/len(chunks_semantic):.0f}')
print(f'\nChunks semanticos encontrados:')
for i, c in enumerate(chunks_semantic[:5]):
    print(f'  [{i}] {c.token_count} tokens: {c.text[:80]}...')

## 3.4 Comparacao: Impacto na Qualidade de Retrieval

In [ ]:
# Comparar qualidade de retrieval por estrategia
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(host='localhost', port=6333)

estrategias = {
    'fixed_size': chunks_fixed,
    'recursive': chunks_recursive,
    'semantic': chunks_semantic,
}

# Indexar cada estrategia em sua propria collection
for nome, chunks in estrategias.items():
    col_name = f'chunk_{nome}'
    if client.collection_exists(col_name):
        client.delete_collection(col_name)
    client.create_collection(col_name, vectors_config=VectorParams(size=384, distance=Distance.COSINE))
    
    textos = [c.text for c in chunks]
    embs = model.encode(textos, normalize_embeddings=True)
    
    points = [
        PointStruct(id=i, vector=embs[i].tolist(), payload={'texto': textos[i]})
        for i in range(len(textos))
    ]
    client.upsert(col_name, points=points)
    print(f'{nome}: {len(chunks)} chunks indexados')

# Queries de teste com respostas esperadas
queries_test = [
    ('Qual e o pipeline do RAG?', 'pipeline'),
    ('O que e chunking semantico?', 'semantic'),
    ('Como avaliar um sistema RAG?', 'ragas'),
    ('Qual e a diferenca entre dense e sparse retrieval?', 'dense sparse'),
]

print('\nResultados de retrieval por estrategia:')
print('='*70)

for query, keyword in queries_test:
    q_vec = model.encode(query, normalize_embeddings=True)
    print(f'\nQuery: {query}')
    
    for nome in estrategias:
        results = client.query_points(f'chunk_{nome}', query=q_vec.tolist(), limit=1, with_payload=True).points
        if results:
            has_keyword = keyword.lower() in results[0].payload['texto'].lower()
            score = results[0].score
            print(f'  {nome:15s}: score={score:.3f} | keyword found={has_keyword}')

In [ ]:
# Visualizacao: distribuicao de tamanhos de chunks
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (nome, chunks) in zip(axes, estrategias.items()):
    token_counts = [c.token_count for c in chunks]
    ax.hist(token_counts, bins=20, color='#3498db', edgecolor='white', alpha=0.8)
    ax.axvline(x=np.mean(token_counts), color='red', linestyle='--', linewidth=2, label=f'Media: {np.mean(token_counts):.0f}')
    ax.set_title(f'{nome}\n({len(chunks)} chunks)', fontweight='bold')
    ax.set_xlabel('Tokens por chunk')
    ax.set_ylabel('Frequencia')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Distribuicao de Tamanho de Chunks por Estrategia', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nObservacoes:')
print('  Fixed-size: distribuicao uniforme (todos similares)')
print('  Recursive: respeitou paragrafos — alguns maiores/menores')
print('  Semantic: alta variancia — dividiu por mudanca de topico')

## Guia de Decisao

```
Tipo de documento?
│
├── Markdown / HTML / estruturado
│   └── Recursive split nos separadores naturais (\n\n, ##, etc.)
│       Ou document-aware chunking respeitando headings
│
├── PDF / DOCX / Texto corrido
│   └── Recursive split (chunk_size=512, overlap=64)
│       E o metodo mais robusto e geral
│
├── Dados sem estrutura / conversacionais
│   └── Semantic chunking (separa por mudanca de topico)
│       Mais caro mas melhor qualidade
│
└── Codigo-fonte
    └── Semantic split por funcao/classe
        Nunca cortar no meio de um bloco de codigo
```

**Parametros recomendados para RAG geral:**
- chunk_size: 256-512 tokens
- overlap: 10-20% do chunk_size
- Estrategia: Recursive Character Split